# 02 Expand Contractions

Reads `processed/01_filtered.csv` and expands contractions ("don't" becomes "do not"). This comes before stop words so that "don't" is not left as "do" and "t".

- The dictionary in `dictionaries/contraction_dict.csv` is used first.
- Anything it misses is handled by general rules: n't, 're, 've, 'll, 'm, 'd.
- Possessives are not contractions: "dog's" becomes "dog" and "dogs'" becomes "dogs".

Output: `processed/02_contractions.csv`, plus `processed/02_contraction_changes.csv` listing every form that was changed.

In [1]:
import pandas as pd
import re
from collections import Counter

In [2]:
data = pd.read_csv("processed/01_filtered.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} rows")

185,494 rows


In [3]:
legacy = pd.read_csv("dictionaries/contraction_dict.csv")
CONTRACTIONS = dict(zip(legacy["contraction"].str.lower(), legacy["expanded"].str.lower()))

# the legacy dictionary leaves some irregular forms out
CONTRACTIONS.update({"won't": "will not", "can't": "can not", "cannot": "can not", "y'all": "you all", "ma'am": "madam"})

WORD = re.compile(r"[a-z]+(?:'[a-z]+)+'?|[a-z]+s'")  # tokens that contain an apostrophe

RULES = [("n't", " not"), ("'re", " are"), ("'ve", " have"), ("'ll", " will"), ("'m", " am"), ("'d", " would")]

changes = Counter()

def expand_token(tok):
    if tok == "one's":
        changes[(tok, "")] += 1     # "one's" is the generic possessive ("one's pleasure"): drop it, not "one"
        return ""
    if tok in CONTRACTIONS:
        new = CONTRACTIONS[tok]
    else:
        new = tok
        for suffix, repl in RULES:
            if tok.endswith(suffix):
                new = tok[: -len(suffix)] + repl
                break
        else:
            if tok.endswith("'s"):
                new = tok[:-2]          # possessive
            elif tok.endswith("s'"):
                new = tok[:-1]          # plural possessive
    if new != tok:
        changes[(tok, new)] += 1
    return new

def expand(text):
    return WORD.sub(lambda m: expand_token(m.group(0)), text)

In [4]:
data["answer_contractions"] = data["answer_clean"].map(expand)

n_changed = (data["answer_contractions"] != data["answer_clean"]).sum()
print(f"{n_changed:,} answers changed")

4,662 answers changed


Every change is saved so it can be reviewed.

In [5]:
log = pd.DataFrame(
    [(a, b, n) for (a, b), n in changes.items()], columns=["original", "expanded", "count"]
).sort_values("count", ascending=False)
log.to_csv("processed/02_contraction_changes.csv", index=False)
log.head(25)

,original,expanded,count
20,one's,,582
11,don't,do not,538
3,can't,can not,375
15,it's,it is,371
1,doesn't,does not,370
22,someone's,someone,218
23,person's,person,191
14,isn't,is not,153
16,you're,you are,107
2,won't,will not,73


In [6]:
# any apostrophes left?
left = data["answer_contractions"].str.findall(r"\w*'\w*").explode().dropna()
print(left.value_counts().head(15))

answer_contractions
'           84
's          28
o'clock     13
80's        10
doesnt't     9
'i           7
70's         6
60's         6
t't          5
't           5
1950's       4
s'mores      4
'to          4
50's         4
1900's       4
Name: count, dtype: int64


In [7]:
out = data[["row_id", "subject_id", "cue", "answer_clean", "answer_contractions"]]
out.to_csv("processed/02_contractions.csv", index=False)
print(f"Saved {len(out):,} rows to processed/02_contractions.csv")
data.loc[data["answer_contractions"] != data["answer_clean"], ["answer_clean", "answer_contractions"]].sample(8, random_state=1)

Saved 185,494 rows to processed/02_contractions.csv


,answer_clean,answer_contractions
178146,"petite, small, opposite of fat, diet, people w...","petite, small, opposite of fat, diet, people w..."
60537,"report relating to one's conduct,a basis or gr...","report relating to conduct,a basis or ground,..."
129659,"fake, misunderstanding, wive's tale","fake, misunderstanding, wive tale"
82236,someone that doesn't tell the truth,someone that does not tell the truth
142749,"two's, ice skating.","two, ice skating."
24917,i don't know this word either but it seems lik...,i do not know this word either but it seems li...
160542,"mountains often have mists, it's in the air, s...","mountains often have mists, it is in the air, ..."
10926,don't know,do not know
